Fitted Q Evaluation: policy value assessment via matched CQL–FQE seed pairs

Following training of five CQL policies and five corresponding FQE critics (each FQE critic trained specifically to estimate the value of the CQL policy sharing its seed, per the target-policy substitution described in Section [X]), we computed the advantage of each CQL policy's chosen actions over the real, logged clinician actions, using that policy's own matched FQE critic: Q_FQE(s, π_CQL(s)) − Q_FQE(s, a_clinician), averaged over all held-out test-set states and normalized by that critic's own interquartile range of Q-values under real clinician actions (consistent with the normalization convention used throughout our SARSA- and CQL-based advantage analyses). Because each FQE critic's value estimates are only theoretically valid for the specific target policy used in its own training (unlike SARSA, a policy-agnostic critic), pairing was performed strictly by seed, verified directly from each model's own saved checkpoint rather than assumed from load order or a shared configuration file.

All five CQL policies showed positive normalized advantage over logged clinician behavior according to their matched FQE critic (mean 0.0079, range 0.0038–0.0116 across seeds), indicating consistent, independently-estimated evidence that the learned policies improve on observed practice. Notably, seed 4 showed the lowest FQE-estimated advantage (0.0038) of the five — the same seed independently flagged in our earlier SARSA-based analysis as anomalous, exhibiting a degenerate vasopressor magnitude distribution (standard deviation saturated at its upper clamp bound in over half of test states) and the weakest SARSA-estimated advantage among the five seeds. That two independent value-estimation approaches — SARSA, an on-policy-bootstrapped critic trained without reference to any specific target policy, and FQE, an off-policy critic fit via iterative Bellman backups under the target policy's own actions — concur on which seed is weakest provides corroborating, method-independent evidence that this seed's reduced quality reflects a genuine property of that trained policy, rather than an artifact specific to either evaluation method.

In [10]:
import sys
import os
from pathlib import Path
import importlib
import json

import numpy as np
import matplotlib.pyplot as plt
import torch


# Optional but very useful in notebooks
%load_ext autoreload
%autoreload 2

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.rl.evaluate_cql_alpha_sweep as eval_cql
import src.rl.evaluate_terminal_reward_models as etm
import src.rl.clinician_baseline as clin_baseline
import src.rl.evaluate_fqe_final_seeds as eval_fqe



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
cohort = "picu"
# health_cohort = "sepsis_picu"
health_cohort = "picu" # dynamic deephit trained on all picu patients, not just sepsis patients

if cohort == "picu":    
    if health_cohort == "picu":
        dynamic_deephit_model_dir = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model"
        print(f"Using dynamic_deephit_model_dir: {dynamic_deephit_model_dir}")
        cql_model_dir = "/home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset"
        print(f"Using cql_model_dir: {cql_model_dir}")
        fqe_model_dir = "/home/jake/Documents/AI_Clinician/models/rl_models/choc/fqe/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset"
        print(f"Using fqe_model_dir: {fqe_model_dir}")

    elif health_cohort == "sepsis_picu":
        dynamic_deephit_model_dir = "/home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/sepsis_picu/final_model"
        print(f"Using dynamic_deephit_model_dir: {dynamic_deephit_model_dir}")
        cql_model_dir = "/home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_restricted_dataset"
        print(f"Using cql_model_dir: {cql_model_dir}")
        fqe_model_dir = "/home/jake/Documents/AI_Clinician/models/rl_models/choc/fqe/split_0/sepsis/intake-vis/picu_patients_health_score_from_restricted_dataset"
        print(f"Using fqe_model_dir: {fqe_model_dir}")


# first, load the dynamic deephit hyperparameters so we can get the state dimension
dynamic_deephit_dir = eval_cql.get_model_dirs(dynamic_deephit_model_dir)[0]
print(f"Dynamic DeepHit model directory: {dynamic_deephit_dir}")
# load results.json from dynamic_deephit_dir
with open(os.path.join(dynamic_deephit_dir, "results.json"), "r") as f:
    dynamic_deephit_results = json.load(f)
state_dim = dynamic_deephit_results['hyperparameters']['d_model']
print(f"State dimension from dynamic deephit hyperparameters: {state_dim}")

fqe_health_score_model_dir = os.path.join(fqe_model_dir, "health_score_change_only")
fqe_health_score_model_dirs = etm.get_model_dirs(fqe_health_score_model_dir)

fqe_models = []
for model_dir in fqe_health_score_model_dirs:
    # load health score sarsa model
    health_score_sarsa_critic, health_score_sarsa_config_manager, health_score_sarsa_hyperparams = etm.load_sarsa_critic(reward_type='health_score', model_dir=model_dir, state_dim=state_dim)
    fqe_models.append((health_score_sarsa_critic, health_score_sarsa_config_manager, health_score_sarsa_hyperparams))


cql_health_score_fixed_alpha_model_dir = os.path.join(cql_model_dir, "health_score_change_only", "fixed_alpha")
cql_health_score_fixed_alpha_model_dirs = eval_cql.get_model_dirs(cql_health_score_fixed_alpha_model_dir)

cql_health_score_fixed_alpha_models = []
for model_dir in cql_health_score_fixed_alpha_model_dirs:
    # load health score sarsa model
    health_score_actor, health_score_critic, health_score_config_manager, health_score_hyperparams = \
        etm.load_actor_and_critic(
            model_dir=model_dir, model_type = 'cql', state_dim=state_dim,
        )

    # print cql value for this model
    print(f"Loaded CQL health-score model from {model_dir}, cql_alpha={health_score_hyperparams['cql_alpha']}")

    cql_health_score_fixed_alpha_models.append((health_score_actor, health_score_critic, health_score_config_manager, health_score_hyperparams))


Using dynamic_deephit_model_dir: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model
Using cql_model_dir: /home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_logprobs/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset
Using fqe_model_dir: /home/jake/Documents/AI_Clinician/models/rl_models/choc/fqe/split_0/sepsis/intake-vis/picu_patients_health_score_from_picu_dataset
No choc directories found in /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model
Found model directory: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model/unknown_cx3-20-1_20260821-202508_49e52257
Dynamic DeepHit model directory: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/five_splits/split_0/sepsis/intake-vis/picu/final_model/unknown_cx3-20-1_20260

# Load data

In [4]:
health_score_config_manager.data_config['unit_stay_ids_path'] = os.path.join(
    health_score_config_manager.get_data_dir, 'cohorts',
    'icu_units',
    f"{health_score_config_manager.unit.lower()}_stay_ids.csv"
)

health_score_config_manager.data_config['age_stay_ids_path'] = os.path.join(
    health_score_config_manager.get_data_dir, 'cohorts',
    'age', 'pediatric_stay_ids.csv'
)

processors = {}
processors['train'], processors['val'], processors['test'] = etm.load_data(health_score_config_manager, health_score_hyperparams)  

using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type
using shifted actions for RL dataset
Using health_score reward type


In [7]:
# real clinician non-zero fluid doses, from training data
fluid_doses = processors['train'].rl_dataset["actions"][:, 0]
nonzero_fluid = fluid_doses[fluid_doses > 1e-4]

# clamp to avoid inf at exact 0/1 boundaries, then invert the sigmoid (logit)
nonzero_fluid_clamped = nonzero_fluid.clamp(1e-4, 1 - 1e-4)
raw_fluid = torch.logit(nonzero_fluid_clamped)

print(f"Fluid: dose-space std={nonzero_fluid.std():.4f}, "
      f"raw-space std={raw_fluid.std():.4f}, raw-space mean={raw_fluid.mean():.4f}")

# same for vaso
vaso_doses = processors['train'].rl_dataset["actions"][:, 1]
nonzero_vaso = vaso_doses[vaso_doses > 1e-4]
nonzero_vaso_clamped = nonzero_vaso.clamp(1e-4, 1 - 1e-4)
raw_vaso = torch.logit(nonzero_vaso_clamped)

print(f"Vaso: dose-space std={nonzero_vaso.std():.4f}, "
      f"raw-space std={raw_vaso.std():.4f}, raw-space mean={raw_vaso.mean():.4f}")

actions = processors['train'].rl_dataset['actions']

fluids = actions[:, 0]
vasopressors = actions[:, 1] 

# get min non-zero fluid and vasopressor doses
min_nonzero_fluid = fluids[fluids > 0].min()
min_nonzero_vasopressor = vasopressors[vasopressors > 0].min()

print(f"Min non-zero fluid dose: {min_nonzero_fluid}")
print(f"Min non-zero vasopressor dose: {min_nonzero_vasopressor}")

clinician_baseline_val = clin_baseline.compute_clinician_baseline(processors['val'].rl_dataset["actions"])
clin_baseline.print_clinician_baseline(clinician_baseline_val, label="val")

clinician_baseline_test = clin_baseline.compute_clinician_baseline(processors['test'].rl_dataset["actions"])
clin_baseline.print_clinician_baseline(clinician_baseline_test, label="test")

Fluid: dose-space std=0.1420, raw-space std=0.6721, raw-space mean=0.1841
Vaso: dose-space std=0.1158, raw-space std=0.6342, raw-space mean=-1.0675
Min non-zero fluid dose: 0.0001319456350756809
Min non-zero vasopressor dose: 0.006044956389814615
=== Clinician baseline (val) ===
  fluid: rate=90.9%, mean=0.4958, nonzero_mean=0.5456
  vaso: rate=8.3%, mean=0.0213, nonzero_mean=0.2569
=== Clinician baseline (test) ===
  fluid: rate=90.5%, mean=0.4900, nonzero_mean=0.5415
  vaso: rate=9.4%, mean=0.0260, nonzero_mean=0.2775


## health_score

In [8]:
training_health_score = processors['train'].dynamic_deephit_predictions["health_score"]
# remove values at indices processors['train'].dynamic_deephit_predictions['index_ptr'][1:] - 1
indices_to_remove = processors['train'].dynamic_deephit_predictions['index_ptr'][1:] - 1
training_health_score = np.delete(training_health_score, indices_to_remove)

val_health_score = processors['val'].dynamic_deephit_predictions["health_score"]
# remove values at indices processors['val'].dynamic_deephit_predictions['index_ptr'][1:] - 1
indices_to_remove = processors['val'].dynamic_deephit_predictions['index_ptr'][1:] - 1
val_health_score = np.delete(val_health_score, indices_to_remove)

test_health_score = processors['test'].dynamic_deephit_predictions["health_score"]
# remove values at indices processors['test'].dynamic_deephit_predictions['index_ptr'][1:] - 1
indices_to_remove = processors['test'].dynamic_deephit_predictions['index_ptr'][1:] - 1
test_health_score = np.delete(test_health_score, indices_to_remove)

training_actions = processors['train'].rl_dataset["actions"]

# fqe evaluation

In [17]:
clinician_actions_test = processors['test'].rl_dataset["actions"]

results = eval_fqe.run_matched_seed_fqe_advantage(
    cql_health_score_fixed_alpha_models,
    fqe_models,
    cql_health_score_fixed_alpha_model_dirs,
    fqe_health_score_model_dirs,
    processors['test'],
    clinician_actions_test,
)

CQL model dirs -> seeds: [('unknown_cx3-20-1_20260913-235751_073683e4', 3), ('unknown_cx3-20-1_20260913-225924_24f16bd0', 1), ('unknown_cx3-20-1_20260913-232805_87a9aaa3', 2), ('unknown_cx3-20-1_20260914-003125_8ef2e012', 4), ('unknown_cx3-20-1_20260913-223211_963944cb', 0)]
FQE model dirs -> seeds: [('unknown_cx3-20-4_20260918-152455_ac14efa2', 0), ('unknown_cx3-20-4_20260918-155016_7b36353f', 1), ('unknown_cx3-20-4_20260918-171242_02a4b9b5', 4), ('unknown_cx3-20-4_20260918-161506_1dabc643', 2), ('unknown_cx3-20-4_20260918-163800_9c127185', 3)]
Matched 5 (CQL, FQE) pairs by checkpoint seed: [0, 1, 2, 3, 4]

=== Seed 0 ===
  Q IQR (clinician actions): 1.1540
  Raw advantage:        mean=0.0084, median=0.0018, std=0.0434
  Normalized advantage: mean=0.0072, median=0.0015

=== Seed 1 ===
  Q IQR (clinician actions): 0.9426
  Raw advantage:        mean=0.0087, median=0.0021, std=0.0479
  Normalized advantage: mean=0.0092, median=0.0022

=== Seed 2 ===
  Q IQR (clinician actions): 0.9877
 

In [15]:
len(fqe_models)

5